# Connection Validation (GasWatch PH)

**Purpose:** Establishes a secure and polite connection to the target server before initiating the BeautifulSoup extraction logic.

**Key Objectives:**
* **Endpoint Definition:** Target both the active price matrix and the historical data pages required for the final JSON structure.
* **Polite Scraping:** Implement standard `User-Agent` headers to simulate legitimate browser traffic, preventing undue strain on the community server.
* **Validation:** Verify `200 OK` network responses to confirm unhindered access to the raw HTML blueprints.

In [1]:
import requests

# Define the exact target URLs based on the site structure
url_prices = "https://gaswatchph.com/"
url_history = "https://gaswatchph.com/price-history"

# Set up the User-Agent disguise for polite scraping
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

print("Initiating connection to GasWatch PH servers...")

# Execute the GET requests
response_prices = requests.get(url_prices, headers=headers)
response_history = requests.get(url_history, headers=headers)

# Verify the connection status
print(f"Prices Page (Home) Status: {response_prices.status_code}")
print(f"History Page Status:     {response_history.status_code}")

if response_prices.status_code == 200 and response_history.status_code == 200:
    print("\nConnection Established Successfully.")
    print("\n--- HTML Preview (Prices Page) ---")
    print(response_prices.text[:300]) 
else:
    print("\nConnection Failed. The server may be blocking the request.")

Initiating connection to GasWatch PH servers...
Prices Page (Home) Status: 200
History Page Status:     200

Connection Established Successfully.

--- HTML Preview (Prices Page) ---
<!DOCTYPE html>
<html lang="en-PH">
<head>
  <meta charset="UTF-8" />
  <meta name="color-scheme" content="light dark" />
  <script>(function(){var t=localStorage.getItem('theme')||(window.matchMedia('(prefers-color-scheme: dark)').matches?'dark':'light');document.documentElement.setAttribute('data-


# Data Extraction & Raw Staging

**Purpose:** Parses the raw, stateful HTML payload into an active memory list (`raw_brand_data`) by dynamically resolving the website's publication cycle and splitting unified UI strings into individual metrics.

**Key Objectives:**
* **Source Date Synchronization:** Extracts the official "Week of" or "As of" text directly from the page body using regular expressions, preventing dependence on local machine timestamps.
* **Dynamic Table Traversal:** Maps the table header (`thead`) dynamically to calculate column alignments, automatically adapting if fuel variants are added or rearranged.
* **Trend & Value Deconstruction:** Deconstructs merged text nodes (e.g., `70.73↓ -8.89`) using a specialized regular expression to isolate the base price, structural trend direction (`UP`/`DOWN`/`STABLE`), and the absolute magnitude of the weekly price change.
* **Decoupled Staging Array:** Packages the parsed fields into a uniform dictionary schema and commits them directly to an active RAM array, separating the extraction phase from downstream storage or validation checks.

In [42]:
import concurrent.futures
from playwright.sync_api import sync_playwright
from bs4 import BeautifulSoup
import re
import uuid
from datetime import datetime

# ==========================================
# 1. THREAD-ISOLATED LIVE DOM FETCH
# ==========================================
def fetch_dynamic_html():
    print("Booting local native browser to bypass OS firewall...")
    with sync_playwright() as p:
        browser = p.chromium.launch(
            headless=True,
            channel="msedge",
            # FINDING 5: --disable-web-security is fine locally. 
            # Swap for '--disable-dev-shm-usage' during AWS Lambda deployment.
            args=['--no-sandbox', '--disable-web-security']
        )
        page = browser.new_page()
        page.goto("https://gaswatchph.com", wait_until="domcontentloaded", timeout=60000)
        page.wait_for_selector("tr:not(.skeleton-row)", timeout=15000)
        html = page.content()
        browser.close()
        return html

# ==========================================
# 2. RUN EXTRACTION AND PARSE TRENDS
# ==========================================
raw_brand_data = []

# FINDING 2: Global Lineage Metadata Generation
INGESTION_RUN_ID = str(uuid.uuid4())
EXTRACTION_TIMESTAMP = datetime.now().isoformat()
DATA_SOURCE = "gaswatchph.com"
PRICE_UNIT = "PHP_per_liter"

# FINDING 2: Slug Generation Utility
def generate_fuel_slug(display_name):
    slug = display_name.lower().strip()
    slug = re.sub(r'[^a-z0-9\s-]', '', slug)
    return re.sub(r'[\s-]+', '-', slug)

with concurrent.futures.ThreadPoolExecutor() as pool:
    rendered_html = pool.submit(fetch_dynamic_html).result()

soup_prices = BeautifulSoup(rendered_html, 'html.parser')
table = soup_prices.find('table', class_='brand-summary-table')
page_text = soup_prices.get_text(separator=' ')

# FINDING 3: ISO 8601 Date Parsing Matrix
date_match = re.search(r'(?:Week of|As of)\s+([A-Za-z]+\s+\d{1,2}(?:\s*[-–]\s*\d{1,2})?,\s+\d{4})', page_text)
if date_match:
    active_week_str = date_match.group(1).strip()
    try:
        # Resolve date ranges (e.g., "June 23 - 29, 2026" -> isolate start date "June 23, 2026")
        if "-" in active_week_str or "–" in active_week_str:
            clean_str = re.sub(r'\s*[-–]\s*\d+', '', active_week_str)
            parsed_date = datetime.strptime(clean_str, "%B %d, %Y")
        else:
            parsed_date = datetime.strptime(active_week_str, "%B %d, %Y")
        week_date_iso = parsed_date.strftime("%Y-%m-%d")
        print(f"Successfully synchronized with database cycle: '{week_date_iso}'")
    except ValueError:
        week_date_iso = datetime.now().strftime("%Y-%m-%d")
        print("Warning: Date parse failed. Falling back to execution timestamp.")
else:
    week_date_iso = datetime.now().strftime("%Y-%m-%d")
    print("Warning: Date banner not located. Falling back to execution timestamp.")

if table:
    headers = [th.text.strip() for th in table.find('thead').find_all('th')]
    tbody = table.find('tbody', id='brandSummaryBody')

    if tbody:
        for row in tbody.find_all('tr'):
            cells = row.find_all('td')

            if len(cells) == len(headers):
                brand_name = cells[0].text.strip()

                for i in range(2, len(cells)):
                    fuel_type = headers[i]
                    raw_cell_text = cells[i].text.strip()

                    if raw_cell_text and "N/A" not in raw_cell_text.upper():
                        
                        match = re.match(r'^([\d\.]+)(?:([↓↑])\s*([\+\-]?[\d\.]+))?', raw_cell_text)

                        if match:
                            base_price = match.group(1)
                            arrow = match.group(2)
                            change_val = match.group(3)

                            direction = "STABLE"
                            if arrow == "↓": direction = "DOWN"
                            elif arrow == "↑": direction = "UP"

                            clean_change = abs(float(change_val)) if change_val else 0.0
                            
                            # FINDING 2 & 7: Enriched Schema without Macro Data
                            raw_brand_data.append({
                                "ingestion_run_id": INGESTION_RUN_ID,
                                "extraction_timestamp": EXTRACTION_TIMESTAMP,
                                "source": DATA_SOURCE,
                                "week_date": week_date_iso,
                                "brand": brand_name,
                                "fuel_display_name": fuel_type,
                                "fuel_type_slug": generate_fuel_slug(fuel_type),
                                "current_price": f"₱{base_price}",
                                "price_unit": PRICE_UNIT,
                                "price_trend": direction,
                                "weekly_change": f"₱{clean_change:.2f}"
                            })

print(f"\nExtraction Complete: {len(raw_brand_data)} trend-enriched records staged.")

print("\n--- Verification Preview (First 2 Records with Expanded Schema) ---")
for record in raw_brand_data[:5]:
    print(record)

Booting local native browser to bypass OS firewall...
Successfully synchronized with database cycle: '2026-06-23'

Extraction Complete: 54 trend-enriched records staged.

--- Verification Preview (First 2 Records with Expanded Schema) ---
{'ingestion_run_id': 'b48336f6-2c15-4e2b-9223-e3bb9c8f0cb9', 'extraction_timestamp': '2026-06-26T20:42:02.278231', 'source': 'gaswatchph.com', 'week_date': '2026-06-23', 'brand': 'Shell', 'fuel_display_name': 'Diesel', 'fuel_type_slug': 'diesel', 'current_price': '₱71.51', 'price_unit': 'PHP_per_liter', 'price_trend': 'DOWN', 'weekly_change': '₱8.11'}
{'ingestion_run_id': 'b48336f6-2c15-4e2b-9223-e3bb9c8f0cb9', 'extraction_timestamp': '2026-06-26T20:42:02.278231', 'source': 'gaswatchph.com', 'week_date': '2026-06-23', 'brand': 'Shell', 'fuel_display_name': 'Prem Diesel', 'fuel_type_slug': 'prem-diesel', 'current_price': '₱81.68', 'price_unit': 'PHP_per_liter', 'price_trend': 'DOWN', 'weekly_change': '₱6.86'}
{'ingestion_run_id': 'b48336f6-2c15-4e2b-92

### Isolated Data Quality Assurance (QA) Gate

**Purpose:** Executes a non-destructive Data Quality Assurance (QA) check on the newly extracted fuel matrix before it moves to permanent storage.

**Key Mechanics:**
* **ELT Non-Destructive Tagging:** Instead of operating as a strict ETL gate that deletes anomalous records, this module adheres to the ELT architecture constraint. It evaluates every row and injects boolean metadata (`qa_passed`) and error arrays (`qa_flags`) directly into the payload, allowing downstream systems (like dbt) to handle the actual filtering logic.
* **Calibrated Sanity Thresholds:** Evaluates core structural integrity, bounds the numeric price float within realistic Philippine fuel limits (₱30.00 to ₱300.00 to safely cover high-octane/kerosene variants), and flags hyper-volatility (price jumps exceeding ₱50.00/week) to catch upstream website typos.
* **Terminal Dashboarding:** Outputs a real-time console summary of passed versus tagged records to provide immediate operational feedback to the developer.

In [43]:
import json

# Ensure that the extraction cell has run first and populated data
if 'raw_brand_data' not in locals() or not raw_brand_data:
    print("Error: 'raw_brand_data' is empty or missing. Please execute the Extraction phase first.")
else:
    print("[System] Initializing ELT Data Quality Assurance (QA) protocols...")

    processed_records = []
    passed_count = 0
    quarantined_count = 0

    for record in raw_brand_data:
        issues = []

        # QA Rule 1: Structural Null Value Check
        if not record.get('brand') or not record.get('fuel_display_name'):
            issues.append("Missing core structural identifiers (Brand or Fuel Type).")

        # QA Rule 2: Price Sanity Thresholds (Ceiling raised to accommodate all fuel types)
        try:
            price_val = float(record['current_price'].replace('₱', '').strip())
            if price_val <= 30.0:
                issues.append(f"Price abnormally low or zero (₱{price_val}).")
            elif price_val >= 300.0:
                issues.append(f"Price exceeds maximum sanity threshold (₱{price_val}).")
        except ValueError:
            issues.append("Current price string formatting contains non-numeric structural errors.")

        # QA Rule 3: Volatility Threshold Check
        try:
            change_val = float(record['weekly_change'].replace('₱', '').strip())
            if change_val > 50.0:
                issues.append(f"Weekly variance value exceeds realistic market volatility (₱{change_val}).")
        except ValueError:
            issues.append("Weekly variance string formatting contains non-numeric structural errors.")

        # ELT METADATA INJECTION (Finding 1): Tag the record instead of deleting/filtering it
        record['qa_flags'] = issues
        record['qa_passed'] = len(issues) == 0
        
        if record['qa_passed']:
            passed_count += 1
        else:
            quarantined_count += 1
            
        processed_records.append(record)

    # --- QA Dashboard Visualization ---
    print("-" * 55)
    print("📊 INTERACTIVE DATA QUALITY ASSURANCE SUMMARY")
    print("-" * 55)
    print(f"Total Raw Records Evaluated: {len(processed_records)}")
    print(f"✅ Passed Validation:        {passed_count}")
    print(f"🚨 Tagged Anomalies:         {quarantined_count}")
    print("-" * 55)

    if quarantined_count > 0:
        print("\n⚠️ WARNING: Corrupted records detected. Tagged as FAILED but retained for ELT storage.")
        for q in processed_records:
            if not q['qa_passed']:
                print(f" - [{q.get('brand')} | {q.get('fuel_display_name')}] FAILED: {q.get('qa_flags')}")
    else:
        print("\n✅ QA Gate Passed perfectly. All records staged for raw storage.")

[System] Initializing ELT Data Quality Assurance (QA) protocols...
-------------------------------------------------------
📊 INTERACTIVE DATA QUALITY ASSURANCE SUMMARY
-------------------------------------------------------
Total Raw Records Evaluated: 54
✅ Passed Validation:        53
🚨 Tagged Anomalies:         1
-------------------------------------------------------

⚠️ WARNING: Corrupted records detected. Tagged as FAILED but retained for ELT storage.
 - [Seaoil | Prem Diesel] FAILED: ['Weekly variance value exceeds realistic market volatility (₱60.99).']


### Persistent Storage & JSON Serialization (ELT Data Lake)

**Purpose:** Finalizes the extraction pipeline by saving the evaluated dataset into a structured, historical data lake architecture, ensuring all raw data is preserved and no historical records are overwritten.

**Key Mechanics:**
* **ELT Complete Retention:** Transitioned the ingestion target from a filtered "valid only" list to `processed_records`. This ensures all data—including the tagged anomalies from the QA gate—is preserved on disk for downstream handling.
* **Raw Partition Routing:** Automatically creates and routes files into a dedicated `data/raw/` folder structure, matching standard data engineering repository layouts and keeping the root directory clean.
* **Dynamic Historical Naming:** Replaces the static, destructive filename with an algorithmic generator. It reads the ISO 8601 date from the dataset (e.g., `2026-06-23`) and compiles a unique output name (e.g., `raw_fuel_prices_2026-06-23.json`). This ensures every weekly run safely accumulates side-by-side for downstream analytics.
* **Unicode Native Dumping:** Enforces `ensure_ascii=False` with mandatory UTF-8 encoding during the `json.dump()` process to perfectly preserve localized currency symbols (₱) without corrupted fallback coding sequences.
* **Automated Fallback Engine:** If the pipeline fails to locate a business date within the record, `datetime.now()` ensures the file is instantly timestamped with the local execution date to prevent a runtime crash.

In [44]:
import json
import os
from datetime import datetime

# Ensure ELT-processed data exists in the runtime environment
if 'processed_records' not in locals() or not processed_records:
    print("Error: No processed records found. Please execute the Data Quality Assurance (QA) cell first.")
else:
    print("[System] Initializing directory checks and ELT persistent storage allocation...")
    
    # Define dedicated raw data lake partition (Finding 4)
    target_directory = os.path.join("data", "raw")
    
    # Extract the dynamic business week date for historical archiving (Finding 4)
    active_date = processed_records[0].get('week_date', datetime.now().strftime("%Y-%m-%d"))
    output_filename = os.path.join(target_directory, f"raw_fuel_prices_{active_date}.json")
    
    try:
        # Automatically create the folder structure if it does not exist yet
        if not os.path.exists(target_directory):
            os.makedirs(target_directory, exist_ok=True)
            print(f"[System] Created new deployment directory: '{target_directory}/'")
            
        # Open file stream inside the target folder with mandatory UTF-8 specification
        with open(output_filename, "w", encoding="utf-8") as json_file:
            json.dump(
                processed_records, 
                json_file, 
                indent=4, 
                ensure_ascii=False
            )
            
        # Verify physical file creation and capture payload metrics
        file_size_bytes = os.path.getsize(output_filename)
        file_size_kb = file_size_bytes / 1024
        
        print("-" * 55)
        print("STORAGE PERSISTENCE SUCCESSFUL (RAW DATA LAKE)")
        print("-" * 55)
        print(f"Target Directory:  {os.path.abspath(target_directory)}")
        print(f"Relative Path:     {output_filename}")
        print(f"Serialized Rows:   {len(processed_records)} records")
        print(f"File Footprint:    {file_size_kb:.2f} KB")
        print("-" * 55)
        print("Pipeline Phase Complete. Asset is secure and raw staging layer closed.")

    except IOError as e:
        print(f"Critical Error: Failed to create directory or write to storage media. Reason: {str(e)}")

[System] Initializing directory checks and ELT persistent storage allocation...
[System] Created new deployment directory: 'data\raw/'
-------------------------------------------------------
STORAGE PERSISTENCE SUCCESSFUL (RAW DATA LAKE)
-------------------------------------------------------
Target Directory:  d:\Downloads\VS Code Projects\ph-fuel-efficiency-app\Notebooks\data\raw
Relative Path:     data\raw\raw_fuel_prices_2026-06-23.json
Serialized Rows:   54 records
File Footprint:    27.94 KB
-------------------------------------------------------
Pipeline Phase Complete. Asset is secure and raw staging layer closed.
